# AFML SLM — Kaggle training

Pretraining for the GQA/DPO project. Run the cells in order.

**Before you start:** Settings (right panel) → Accelerator → **GPU T4 x2** (or P100),
and Internet → **On**.

**Session limits.** A GPU session is killed at 12 hours and the weekly quota is
about 30 GPU-hours. Checkpoints are written every `checkpoint_interval` steps,
so a killed run resumes from the last one — verified by
`scripts/test_resume.py`, which passes bit-exact.

**Across sessions:** click *Save Version* → *Save & Run All* when a run finishes.
That persists `/kaggle/working`. To continue later, attach this notebook's output
as an input dataset and set `PREV_OUTPUT` in section 6.

## 1. Code

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

REPO   = "https://github.com/Shikshith05/AFML_miniproject.git"
BRANCH = "main"
ROOT   = Path("/kaggle/working/AFML_miniproject")

if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(ROOT)],
    check=True,
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

print("at", Path.cwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
# torch and numpy ship with Kaggle; only these three are missing
!pip install -q tokenizers datasets ftfy

## 2. GPU check

In [ ]:
import torch

print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU, then rerun.")

p = torch.cuda.get_device_properties(0)
cc = (p.major, p.minor)
print("device         ", p.name)
print("memory         ", f"{p.total_memory / 1e9:.1f} GB")
print("compute cap    ", f"{cc[0]}.{cc[1]}")

# torch.cuda.is_bf16_supported() returns True on Turing (T4, cc 7.5) because
# bf16 can be emulated -- but there are no bf16 tensor cores before Ampere
# (cc 8.0), so emulated bf16 is SLOWER than fp16 for no accuracy benefit.
native_bf16 = cc >= (8, 0)
precision = "bf16" if native_bf16 else "fp16"
print("native bf16    ", native_bf16)
print()
print(f"-> use precision '{precision}'")
if not native_bf16:
    print("   (fp16 + GradScaler; this GPU has no bf16 tensor cores)")

## 3. Data

Builds `data/train.bin` and `data/val.bin` from TinyStories using the tokenizer
committed in the repo.

A full build costs about 40 minutes of session time. Do it once, *Save Version*,
then attach that output in later sessions and set `DATA_FROM` to skip it.

In [ ]:
from pathlib import Path
import shutil

DATA_FROM = None    # e.g. "/kaggle/input/afml-slm-data/data"
LIMIT     = None    # None = full dataset; e.g. 200000 to go faster

data_dir = Path("data")

if DATA_FROM and Path(DATA_FROM).exists():
    data_dir.mkdir(exist_ok=True)
    for f in Path(DATA_FROM).glob("*"):
        shutil.copy(f, data_dir / f.name)
    print("copied prepared data from", DATA_FROM)
elif (data_dir / "train.bin").exists():
    print("data already present")
else:
    cmd = "python scripts/prepare_data.py"
    if LIMIT:
        cmd += f" --limit {LIMIT}"
    print(cmd)
    get_ipython().system(cmd)

for f in sorted(data_dir.glob("*.bin")):
    print(f"{f.name:>12}  {f.stat().st_size / 1e6:>8.1f} MB")

## 4. Probe

Finds the largest batch size that fits, measures throughput, and turns a
wall-clock budget into `max_steps`.

Run this **once per GPU type**. Use the same numbers for both the MHA and the
GQA run — matched conditions are what make the ablation valid.

In [ ]:
!python scripts/probe_batch_size.py --preset gqa --budget-hours 3

## 5. Train

Fill in `MICRO_BS`, `ACCUM` and `MAX_STEPS` from the probe output above.

Run MHA first, then GQA, changing only `PRESET`. Everything else stays
identical, seed included.

In [ ]:
PRESET    = "mha"     # "mha" first, then "gqa"
MICRO_BS  = 16        # <- from the probe
ACCUM     = 8         # <- from the probe
MAX_STEPS = 20000     # <- from the probe
PRECISION = "fp16"    # "bf16" if section 2 said it is supported

from slm.model.config import gqa_base, mha_baseline

cfg = {"mha": mha_baseline, "gqa": gqa_base}[PRESET]()
cfg.run_name               = f"{PRESET}_base"
cfg.train.micro_batch_size = MICRO_BS
cfg.train.grad_accum_steps = ACCUM
cfg.train.max_steps        = MAX_STEPS
cfg.train.precision        = PRECISION

print(cfg.summary())

In [ ]:
from slm.model.transformer import Transformer
from slm.data.tinystories import make_batch_provider
from slm.training.pretrain import pretrain

model     = Transformer(cfg.model)
get_batch = make_batch_provider(cfg)

# resume=True picks up the newest checkpoint in checkpoints/<run_name>/
pretrain(model, cfg, get_batch, resume=True)

## 6. Resuming after a killed session

1. *Save Version* → *Save & Run All* so `/kaggle/working` is kept
2. In the new session: *Add Input* → *Notebook Output* → pick this notebook
3. Set `PREV_OUTPUT` below and run this cell, then rerun section 5 —
   `resume=True` finds the checkpoint and continues

In [ ]:
PREV_OUTPUT = None    # e.g. "/kaggle/input/afml-slm-mha/AFML_miniproject/checkpoints"

from pathlib import Path
import shutil

if PREV_OUTPUT and Path(PREV_OUTPUT).exists():
    dest = Path("checkpoints")
    dest.mkdir(exist_ok=True)
    for run in Path(PREV_OUTPUT).iterdir():
        if run.is_dir():
            shutil.copytree(run, dest / run.name, dirs_exist_ok=True)
            ckpts = sorted(p.name for p in (dest / run.name).glob("*.pt"))
            print(run.name, "->", ckpts[-3:] if ckpts else "none")
else:
    print("PREV_OUTPUT not set -- starting fresh")

## 7. Results

`results/<run_name>/metrics.jsonl` holds every logged step. Download it from the
Output tab and commit it to the repo — those numbers go into the paper.

In [ ]:
from pathlib import Path
from slm.training.utils import read_metrics

for m in sorted(Path("results").glob("*/metrics.jsonl")):
    rows  = list(read_metrics(m))
    evals = [r for r in rows if r.get("event") == "eval"]
    line  = f"{m.parent.name:>12}  {len(rows):>5} rows"
    if evals:
        line += (f"   last val_loss {evals[-1]['val_loss']:.4f}"
                 f"   ppl {evals[-1]['val_ppl']:.2f}")
    print(line)